# Research Models V2: stronger teachers for SpikeDistill

**Question:** Do published CNN and attention BiGRU architecture choices yield better teachers, and how much of that accuracy transfers to a fixed 64-neuron spiking student?

- **Li-style DCNN:** four temporal convolutions with 10 filters and kernel 10, a final kernel-3 convolution, and a 100-unit regression head. See [Li, Ding & Sun (2018)](https://doi.org/10.1016/j.ress.2017.11.021).
- **Attention BiGRU:** two bidirectional GRU layers, self-attention across observed cycles, masked pooling and a latest-reading shortcut. See [Fu et al. (2026)](https://doi.org/10.1038/s41598-026-46237-y). We retain our train-fitted sensor pipeline rather than recreating the authors' health-index and alignment pipeline.
- **SNN variants:** ground truth alone, CNN teacher, BiGRU teacher, or an equal average of their frozen predictions. Same SNN architecture and training budget; distillation uses 60% true-label MSE and 40% teacher MSE.

All models train on capped RUL scaled by the cap. Checkpoints are selected by raw RMSE on disjoint validation engines. The test stage opens one last observed window per official test engine only after checking the frozen checkpoint manifest. Raw and capped targets are reported separately, in cycles, using the same unclipped predictions. Published figures are external references.

In [ ]:
from pathlib import Path
import subprocess
import sys
import torch
from google.colab import drive

drive.mount("/content/drive")
project = Path("/content/drive/MyDrive/ML projec/SpikeDistill-RUL")
data_root = project / "artifacts/data"
run_root = project / "colab_runs/research_teachers_v2"
repo = Path("/content/ML3103-project-v2")
if not (repo / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/abhav-arora/ML3103-project.git", str(repo)], check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True)

subsets = ["FD001", "FD002", "FD003", "FD004"]
seeds = [42]
teacher_epochs, student_epochs = 50, 30
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → T4 GPU, then rerun setup.")
for subset in subsets:
    if not (data_root / subset / "train.npz").is_file():
        raise FileNotFoundError(data_root / subset / "train.npz")
command = [sys.executable, "-u", "-m", "research_extensions.v2.run",
           "--data-root", str(data_root), "--output", str(run_root),
           "--subsets", *subsets, "--seeds", *map(str, seeds), "--device", "cuda",
           "--teacher-epochs", str(teacher_epochs), "--student-epochs", str(student_epochs)]
print(torch.cuda.get_device_name(), "| Saving checkpoints to", run_root)

## 1. Train and freeze checkpoint selections

Rerun this cell with the same settings after a runtime disconnect. The latest complete epoch for each model is restored from Drive.

In [ ]:
subprocess.run([*command, "--phase", "train"], cwd=repo, check=True)

## 2. Evaluate on official test engines

In [ ]:
subprocess.run([*command, "--phase", "test"], cwd=repo, check=True)

## 3. Compare each teacher and its student

A negative student-minus-supervised RMSE indicates a benefit from teacher guidance. A negative student-minus-teacher RMSE means the student beat its teacher. The original models are shown on the same raw RUL test protocol.

In [ ]:
import pandas as pd
from IPython.display import display

scores = pd.read_csv(run_root / "research_metrics.csv")
pairs = pd.read_csv(run_root / "teacher_student_pairs.csv")
reference = pd.read_csv(run_root / "original_raw_reference.csv")
metrics = ["RMSE", "MAE", "MSE", "RAE_pct", "R2", "cmapss_score", "bias_cycles"]
raw = scores.query("split == 'test' and target == 'raw'")
display(pairs.query("split == 'test' and target == 'raw'").round(3))
display(raw[["subset", "seed", "model", *metrics]].round(3))
display(scores.query("split == 'test' and target == 'capped'")[["subset", "seed", "model", *metrics]].round(3))

combined = pd.concat([raw, reference.assign(model=lambda t: "original_" + t.model)])
rmse = combined.pivot(index=["subset", "seed"], columns="model", values="RMSE")
v1 = pd.read_csv(repo / "research_extensions/results/teacher_student/colab/research_metrics.csv")
v1 = v1.query("split == 'test' and target == 'raw'").assign(model=lambda t: "v1_" + t.model)
rmse = rmse.join(v1.pivot(index=["subset", "seed"], columns="model", values="RMSE"))
display(rmse.round(3))
rmse.to_csv(run_root / "test_rmse_v2_v1_original.csv")

In [ ]:
import matplotlib.pyplot as plt

labels = {"li_dcnn": "Li-style CNN", "attention_bigru": "Attention BiGRU",
          "teacher_average": "Teacher average", "snn_supervised": "SNN supervised",
          "snn_from_li_dcnn": "SNN from CNN", "snn_from_bigru": "SNN from BiGRU",
          "snn_from_both": "SNN from both", "original_spikedistill": "Original SpikeDistill"}
columns = [name for name in labels if name in rmse]
axes = rmse[columns].rename(columns=labels).plot.bar(
    figsize=(13, 4), rot=0, ylabel="Test RMSE (cycles)", title="Official test sets: raw RUL")
axes.legend(ncol=4, fontsize=8)
axes.figure.tight_layout()
axes.figure.savefig(run_root / "test_rmse_v2.png", dpi=180)
plt.show()